In [36]:
from embedder import Embedder
import numpy as np
from tqdm import tqdm
from minsearch import VectorSearch 

embed = Embedder()

Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

- -0.31
- -0.02 +++
- 0.12
- 0.44

In [ ]:
homework_query = 'How does approximate nearest neighbor search work?'

homework_vector = embed.encode(homework_query)

In [2]:
homework_vector[0]

np.float64(-0.02058200593003704)

Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

- 0.07
- 0.37 +++
- 0.68
- 0.92

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [11]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [19]:
sqlitesearch_text_lesson_7 = [doc['content'] for doc in documents \
    if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md' ][0]

In [20]:
sqlitesearch_vector =  embed.encode(sqlitesearch_text_lesson_7)

In [21]:
homework_vector.dot(sqlitesearch_vector)

np.float64(0.36107029062443674)

Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:


We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

Which file does the highest-scoring chunk belong to (its filename)?

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- 02-vector-search/lessons/07-sqlitesearch-vector.md +++
- 02-vector-search/lessons/09-onnx-embedder.md

In [22]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [38]:

X = []

for chunk in tqdm(chunks):
    vector = embed.encode(chunk['content'])
    X.append(vector)

100%|██████████| 295/295 [00:11<00:00, 26.67it/s]


In [42]:
chunk

{'start': 1000,
 'content': 'c.\n\n):\n\n1. Split it into chunks\n2. Evaluate the same way as multiple articles\n3. You can use `youtube-transcript-api` to get transcripts\n   programmatically\n\n## Book or very long content\n\nFor books and other long-form content, apply this strategy:\n\n1. Treat each chapter or section as a separate document\n2. Experiment with different chunking strategies\n3. Use LLM-as-a-Judge to compare approaches\n\n## Images and slides\n\nVisual content can be processed as follows:\n\n1. Describe images using an LLM like GPT-4o-mini\n2. Each image is a separate document\n3. For slide decks: deck = document, slide = chunk\n4. You can also use CLIP embeddings for direct image search\n\n## Smart chunking with LLMs\n\nInstead of splitting by character count or paragraph breaks, you\ncan use an LLM to find logical boundaries:\n\n1. Give the LLM the full text and ask it to split into logical\n   blocks\n2. Then ask it to name each block\n3. Each block becomes a chun

In [ ]:
X = np.array(X)
X.shape

In [34]:
scores = X.dot(homework_vector)
idx = np.argmax(scores)

chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- 04-evaluation/lessons/05-search-metrics.md +++
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

In [83]:
vector_search_query = 'What metric do we use to evaluate a search engine?'
chunk_text = [chunk['content'] for chunk in chunks]

vindex = VectorSearch() 
vindex.fit(X,chunks)

In [84]:
vector_search_vector = embed.encode(vector_search_query)

results = vindex.search(vector_search_vector, num_results=5) # we search for the most similar documents
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

- 02-vector-search/lessons/01-intro.md
- 02-vector-search/lessons/02-embeddings.md
- 02-vector-search/lessons/08-pgvector.md +++
- 03-orchestration/lessons/05-rag.md

In [ ]:
import sys
sys.path.insert(0, '..')

In [85]:
from minsearch import Index

index = Index(text_fields=['content'])
index.fit(chunks)

In [92]:
postgres_query  = 'How do I store vectors in PostgreSQL?'

results_posgre = index.search(postgres_query, num_results=5)
top_5_postgre =  [res['filename'] for res in results_posgre]
top_5_postgre

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [93]:
vector_search_vector = embed.encode(postgres_query)

results_vector = vindex.search(vector_search_vector, num_results=5) 

top_5_vector = [res['filename'] for res in results_vector]
top_5_vector

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [94]:
set([name for name in top_5_vector  if name not in top_5_postgre])

{'02-vector-search/lessons/08-pgvector.md'}

Q6. Hybrid search
Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.


Now run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

results = rrf([vector_results, text_results])
Which file is ranked first after RRF?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/13-function-calling.md +++
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md


Notice that this file isn't first in either search on its own - it wins because it ranks high in both.

In [91]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [104]:
hybrid_query = "How do I give the model access to tools?"
hybrid_query_vector = embed.encode(hybrid_query)
results_posgre = index.search(hybrid_query)
results_vector = vindex.search(hybrid_query_vector) 

In [105]:
results_hybrid = rrf([results_vector, results_posgre])

In [107]:
results_hybrid[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'